# Demo — EPINUC nucleosome/PTM colocalization per sample

This notebook uses the importable `epinuc_colocalization.py` module. Give it a sample id
(or a list) and it runs the full pipeline and writes **one combined CSV with a single row
per sample** — the final cumulative count of each type (nucleosomes, R PTMs, B PTMs, and the
R / B / triple colocalizations) across all fields of view.

Requirements: `epinuc_colocalization.py` next to this notebook, and the ND2 files in a
folder (default `./T50_20260225`). Install deps once:
`pip install nd2 numpy pandas matplotlib scipy scikit-image scikit-learn tifffile tqdm joblib`.

In [ ]:
import epinuc_colocalization as ep

DATA_DIR = "./T50_20260225"      # TODO: folder with your ND2 files
OUTPUT_DIR = "./demo_output"     # where the summary CSV is written

# Which samples are present? (parses TS<timepoint>_<sample>.nd2)
by_sample = ep.group_files_by_sample(ep.list_nd2_files(DATA_DIR))
print("Available samples ->", {s: len(v) for s, v in sorted(by_sample.items())})

## 1. One sample (quick test on the first 3 FOVs)

`scenes=range(3)` keeps this fast. Use `scenes=None` to process every field of view.

In [ ]:
df_one = ep.run_samples(
    1,                       # sample id (int) — or a list, see below
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    scenes=range(3),         # TODO: set to None for all FOVs
    n_jobs=ep.N_JOBS,        # -2 = all but one core; 1 = serial
)
df_one

## 2. Several samples at once

Pass a list of ids. Each sample becomes one row in the returned table and in
`OUTPUT_DIR/samples_summary.csv`.

In [ ]:
df = ep.run_samples(
    [1, 2, 3],               # TODO: your sample ids, e.g. list(by_sample)
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    scenes=range(3),         # TODO: None for the full run
)
df

## 3. Full production run & extras

- **All FOVs:** set `scenes=None` (a full 49-FOV sample takes ~0.5–1 min on many cores).
- **Also keep the detailed tables** (per-spot, per-bead, colocalization events, transforms,
  per-image and cumulative counts): pass `write_details=True` — they go to
  `OUTPUT_DIR/sample_<id>/`.
- **Command line** (no notebook needed):
  `python epinuc_colocalization.py 1 2 3 --data-dir T50_20260225 --output-dir demo_output`

Tuning knobs (detection thresholds, colocalization radius, channel map, etc.) live as
constants in `epinuc_colocalization.py`; override per call via `run_samples(..., channel_map=...)`
or edit the module. See the main analysis notebook for the QC plots and threshold-tuning
helpers used to choose them.

In [ ]:
# Full run for every available sample, keeping the detailed CSVs too (uncomment):
# df_all = ep.run_samples(sorted(by_sample), data_dir=DATA_DIR, output_dir=OUTPUT_DIR,
#                         scenes=None, write_details=True)
# df_all